# Spatial Travel Intelligence Prototype

MVP focus: ingest Google Maps pins, cluster into travel zones, estimate transit burden, and generate simple itinerary suggestions using weighted scoring.

Phases 1–2 are fully implemented. Phase 3 uses a simple scoring engine (no fatigue simulation yet).

In [ ]:
# Ensure optional visualization/export packages are installed in the current Python environment
import importlib, subprocess, sys
def ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
        print(f'Package available: {name}')
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

# Packages useful for visualization and scoring (optional)
ensure('folium')
ensure('networkx')
ensure('scikit-learn', 'sklearn')
ensure('pyyaml', 'yaml')

In [ ]:
from pathlib import Path
import sys

# Ensure the repository root is on sys.path so `ancestry_code` is importable
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd

# Import core modules (visualization is optional -- may require folium)
from ancestry_code.src import (
    GeoClusterer,
    GeoClustererConfig,
    ItineraryEngine,
    ItineraryEngineConfig,
    PinProcessor,
    PinProcessorConfig,
    RouteOptimizer,
    RouteOptimizerConfig,
    SyntheticDataConfig,
    TransitBurdenConfig,
    generate_synthetic_pins,
)

# Optional imports: MapVisualizer requires `folium`; import lazily and handle missing dependency
try:
    from ancestry_code.src import ItineraryExporter, MapVisualizer  # MapVisualizer uses folium
    HAS_VISUAL = True
except Exception as exc:
    ItineraryExporter = None
    MapVisualizer = None
    HAS_VISUAL = False
    print("Optional visualization/export modules not available:", exc)

CONFIG_PATH = ROOT / "travel" / "config" / "travel_config.yaml"
OVERRIDE_PATH = ROOT / "travel" / "config" / "cluster_override.yaml"
OUTPUT_DIR = ROOT / "travel" / "outputs"

In [ ]:
def check_geo_stack():
    missing = []
    for package in ["geopandas", "shapely", "fiona", "pyproj"]:
        try:
            __import__(package)
        except ImportError:
            missing.append(package)
    return missing

missing_geo = check_geo_stack()
if missing_geo:
    print("Missing geo packages:", ", ".join(missing_geo))
    print("If installs fail, consider using conda-forge for geopandas stack.")
else:
    print("Geospatial stack is available.")

In [ ]:
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)
config

In [ ]:
processor_config = PinProcessorConfig(
    default_visit_duration_hours=float(config["pins"]["default_visit_duration_hours"]),
    category_durations=config["pins"]["category_durations"],
    duplicate_distance_km=float(config["validation"]["duplicate_distance_km"]),
)
pin_processor = PinProcessor(processor_config)

geojson_path = Path(config["data"].get("geojson_path") or "")
kml_path = Path(config["data"].get("kml_path") or "")

# Only treat the path as a source if it is an actual file (not a directory).
if geojson_path and geojson_path.is_file():
    pins = pin_processor.load_geojson(geojson_path)
elif kml_path and kml_path.is_file():
    pins = pin_processor.load_kml(kml_path)
else:
    print("No pin file provided. Generating synthetic sample pins.")
    pins = generate_synthetic_pins(SyntheticDataConfig())

pins.head()

In [ ]:
is_valid, violations = pin_processor.validate_coordinates(pins)
duplicate_pairs = pin_processor.detect_duplicates(pins)

print("Valid coordinates:", is_valid)
if violations:
    print("Violations:")
    for violation in violations:
        print(" -", violation)

print("Duplicate pairs:", duplicate_pairs[:5])
pins.describe(include="all").T

In [ ]:
clusterer = GeoClusterer(
    GeoClustererConfig(
        eps_km=float(config["clustering"]["eps_km"]),
        min_samples=int(config["clustering"]["min_samples"]),
    )
)

clustered_pins = clusterer.cluster_by_distance(pins)
cluster_summary = clusterer.build_cluster_summary(clustered_pins)

override_result = clusterer.apply_overrides(clustered_pins, cluster_summary, OVERRIDE_PATH)
clustered_pins = override_result["pins"]
cluster_summary = override_result["summary"]

cluster_order = clusterer.optimize_cluster_order(cluster_summary)
cluster_summary

In [ ]:
transit_config = TransitBurdenConfig(multipliers=config["transit_burden"]["multipliers"])
route_optimizer = RouteOptimizer(
    RouteOptimizerConfig(
        average_speed_kph=float(config["transit_burden"]["average_speed_kph"]),
        transit_burden=transit_config,
    )
)

routes = route_optimizer.estimate_cluster_routes(cluster_summary, cluster_order)

engine = ItineraryEngine(
    ItineraryEngineConfig(
        max_pins_per_day=int(config["scoring"]["max_pins_per_day"]),
        score_weights=config["scoring"]["weights"],
    )
)
itinerary = engine.generate_simple_itinerary(clustered_pins, cluster_order, routes)

pd.DataFrame([day.__dict__ for day in itinerary.days])

In [ ]:
if HAS_VISUAL and MapVisualizer is not None:
    visualizer = MapVisualizer(OUTPUT_DIR)
    cluster_map_path = visualizer.plot_clusters(clustered_pins, cluster_summary)
    itinerary_map_path = visualizer.plot_itinerary(clustered_pins, itinerary)
    cluster_map_path, itinerary_map_path
else:
    print("Visualization disabled: install 'folium' to enable maps.")
    cluster_map_path = None
    itinerary_map_path = None
    cluster_map_path, itinerary_map_path

In [ ]:
if ItineraryExporter is not None:
    exporter = ItineraryExporter(OUTPUT_DIR)
    export_paths = exporter.export(itinerary)
    export_paths
else:
    print("Export skipped: ItineraryExporter unavailable (install optional dependencies).")
    export_paths = {}
    export_paths

In [ ]:
category_counts = pins["category"].value_counts().to_dict()
prep_summary = {
    "dominant_categories": category_counts,
    "language_prep_note": "Placeholder: language detection will be added in Phase 5.",
}
prep_summary